<a href="https://colab.research.google.com/github/e23383/Statistical-Learning-e23383/blob/main/Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
import pandas as pd
import numpy as np
import io
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.experimental import enable_iterative_imputer  # Enables MICE
from sklearn.impute import KNNImputer, IterativeImputer
from typing import List, Optional, Union, Dict, Any
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# MAJESTIC VISUAL PALETTES & CONFIGURATION
# ==========================================
class LuxuryTheme:
    """Enterprise-level custom aesthetic palette for premium reports."""
    BACKGROUND = "#0F1016"       # Dark Slate Luxury Caviar
    PAPER_BG = "#161722"         # Deep Tech Blue
    PRIMARY = "#FFD700"          # Majestic Gold
    SECONDARY = "#FF6B6B"        # Rose Crimson
    ACCENT = "#4D96FF"           # Neon Sapphire
    MUTED = "#6B728E"            # Platinum Gray
    GRID = "#252836"             # Deep Grid highlight

    SEQUENTIAL = ["#1A1B2F", "#3B3D62", "#5C5E96", "#8385C8", "#A9ACFC", "#FFD700"]
    DIVERGING = px.colors.diverging.RdBu_r

    @classmethod
    def apply_layout(cls, fig: go.Figure, title: str) -> go.Figure:
        """Transforms a standard plot into an award-winning executive visual."""
        fig.update_layout(
            title=dict(
                text=f"<b>{title.upper()}</b>",
                font=dict(family="Courier New, monospace", size=18, color=cls.PRIMARY),
                x=0.05, y=0.95
            ),
            font=dict(family="Arial, sans-serif", color="#FFFFFF"),
            paper_bgcolor=cls.BACKGROUND,
            plot_bgcolor=cls.PAPER_BG,
            colorway=[cls.PRIMARY, cls.SECONDARY, cls.ACCENT, cls.MUTED],
            margin=dict(t=80, b=50, l=50, r=50),
            xaxis=dict(
                gridcolor=cls.GRID,
                zerolinecolor=cls.GRID,
                title_font=dict(color=cls.MUTED),
                tickfont=dict(color=cls.MUTED)
            ),
            yaxis=dict(
                gridcolor=cls.GRID,
                zerolinecolor=cls.GRID,
                title_font=dict(color=cls.MUTED),
                tickfont=dict(color=cls.MUTED)
            )
        )
        return fig


class PlottingMethods:

    @staticmethod
    def create_bar_chart(data: pd.DataFrame, x: str, y: Optional[str] = None, title: str = "Bar Chart", xlabel: Optional[str] = None, ylabel: Optional[str] = None) -> str:
        fig = px.bar(data, x=x, y=y, title=title, labels={x: xlabel or x, y: ylabel or y})
        fig = LuxuryTheme.apply_layout(fig, title)
        return fig.to_html(full_html=False)

    @staticmethod
    def create_pie_chart(data: pd.DataFrame, names: str, values: str, title: str = "Pie Chart") -> str:
        fig = px.pie(data, names=names, values=values, hole=0.4)
        fig = LuxuryTheme.apply_layout(fig, title)
        fig.update_traces(textinfo='percent+label', marker=dict(line=dict(color=LuxuryTheme.BACKGROUND, width=2)))
        return fig.to_html(full_html=False)

    @staticmethod
    def create_histogram(data: pd.DataFrame, column: str, bins: int = 30, title: str = "Histogram", xlabel: Optional[str] = None) -> str:
        fig = px.histogram(data, x=column, nbins=bins, labels={column: xlabel or column})
        fig = LuxuryTheme.apply_layout(fig, title)
        fig.update_traces(marker=dict(line=dict(color=LuxuryTheme.BACKGROUND, width=0.5)))
        return fig.to_html(full_html=False)


class DataInspector:

    def __init__(self):
        self.data: Optional[pd.DataFrame] = None
        self.numeric_cols: List[str] = []
        self.categorical_cols: List[str] = []
        self.original_data: Optional[pd.DataFrame] = None

    def upload_data(self) -> pd.DataFrame:
        print("⚡ Please upload your raw pipeline CSV file:")
        uploaded = files.upload()

        for filename in uploaded.keys():
            self.data = pd.read_csv(io.BytesIO(uploaded[filename]))
            self.original_data = self.data.copy()
            print(f"✅ Successfully loaded: {filename}")
            print(f"📊 Dimensional Matrix Shape: {self.data.shape}")
            self._identify_column_types()
            return self.data

    def _identify_column_types(self) -> None:
        if self.data is not None:
            self.numeric_cols = self.data.select_dtypes(include=[np.number]).columns.tolist()
            self.categorical_cols = self.data.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

    def _convert_garbage_to_nan(self) -> None:
        garbage_values = ['?', 'n/a', 'NULL', '', 'None', 'null', 'NaN', 'nan', 'unknown', 'Unknown']
        self.data = self.data.replace(garbage_values, np.nan)

    def _auto_type_correction(self) -> None:
        for col in self.data.columns:
            if self.data[col].dtype == 'object':
                try:
                    converted = pd.to_numeric(self.data[col])
                    if not converted.isna().all():
                        self.data[col] = converted
                        print(f"✨ Auto-Engineered Type Correction: Converted '{col}' to Numerical Pipeline.")
                except:
                    pass
        self._identify_column_types()

    def data_summary(self) -> None:
        print("\n" + "👑 " * 20)
        print("                       MAJESTIC DATA SUMMARY REPORT")
        print("👑 " * 20)
        print(f"📈 Total Rows: {self.data.shape[0]:,} | Total Columns: {self.data.shape[1]:,}")
        print(f"🔢 Numeric Elements: {len(self.numeric_cols)} | 🔤 Categorical Features: {len(self.categorical_cols)}")

        print("\n📥 METADATA OBSERVATION MATRIX (FIRST 5 ROWS):")
        print(self.data.head(5).to_markdown()) # Looks way cleaner than raw printing

        print("\n🧮 SCIENTIFIC NUMERICAL SYNOPSIS:")
        if self.numeric_cols:
            print(self.data[self.numeric_cols].describe().T.to_markdown())
        else:
            print("None Found.")

        print("\n🔖 HIGH-CARDINALITY CATEGORICAL PROFILE:")
        for col in self.categorical_cols:
            print(f"  • Feature [{col}] ➡️ Unique Vectors: {self.data[col].nunique()}")

    def handle_missing_values(self, strategy: str = 'median', target_col: Optional[str] = None) -> pd.DataFrame:
        print(f"\n🛠️ Executing Advanced Imputation Paradigm: [{strategy.upper()}]")
        self._convert_garbage_to_nan()
        self._auto_type_correction()

        if not self.data.isna().sum().sum():
            print("🚀 Dataset is perfectly clean. Skip Imputation Strategy.")
            return self.data

        # Advanced ML Data Imputations
        if strategy in ['knn', 'mice']:
            # Drop categorical strings temporarily for ML imputers if not encoded
            temp_num = self.data[self.numeric_cols].copy()
            if strategy == 'knn':
                imputer = KNNImputer(n_neighbors=5)
            else:
                imputer = IterativeImputer(random_state=42)

            imputed_array = imputer.fit_transform(temp_num)
            self.data[self.numeric_cols] = pd.DataFrame(imputed_array, columns=self.numeric_cols, index=self.data.index)

            # Categorical fallback to mode
            for col in self.categorical_cols:
                if self.data[col].isna().sum() > 0:
                    mode_val = self.data[col].mode().iloc[0] if not self.data[col].mode().empty else 'Unknown'
                    self.data[col] = self.data[col].fillna(mode_val)
        else:
            # Baseline strategies mapping
            for col in self.data.columns:
                if self.data[col].isna().sum() > 0:
                    if col in self.numeric_cols:
                        if strategy == 'mean':
                            val = self.data[col].mean()
                        elif strategy == 'median':
                            val = self.data[col].median()
                        else:
                            val = self.data[col].mode().iloc[0] if not self.data[col].mode().empty else 0
                        self.data[col] = self.data[col].fillna(val)
                    else:
                        val = self.data[col].mode().iloc[0] if not self.data[col].mode().empty else 'Unknown'
                        self.data[col] = self.data[col].fillna(val)

        print(f"💎 Matrix Cleansed. Unresolved Nulls Remaining: {self.data.isna().sum().sum()}")
        return self.data

    def remove_duplicates(self) -> pd.DataFrame:
        initial = len(self.data)
        self.data = self.data.drop_duplicates()
        print(f"🗑️ Cleaned {initial - len(self.data)} Redundant Rows From Memory Pool.")
        return self.data

    def handle_outliers(self, columns: Optional[List[str]] = None, method: str = 'flag', iqr_multiplier: float = 1.5) -> pd.DataFrame:
        if columns is None:
            columns = self.numeric_cols

        outlier_flags = pd.Series([False] * len(self.data), name='is_outlier', index=self.data.index)

        for col in columns:
            if col in self.numeric_cols:
                Q1 = self.data[col].quantile(0.25)
                Q3 = self.data[col].quantile(0.75)
                IQR = Q3 - Q1
                lower = Q1 - iqr_multiplier * IQR
                upper = Q3 + iqr_multiplier * IQR

                col_outliers = (self.data[col] < lower) | (self.data[col] > upper)
                outlier_flags |= col_outliers

        if method == 'flag':
            self.data['is_outlier'] = outlier_flags.astype(int)
            print(f"🚩 Outlier Scanner Complete: Vector Flagged {outlier_flags.sum()} out-of-bounds metrics.")
        elif method == 'delete':
            initial = len(self.data)
            self.data = self.data[~outlier_flags]
            print(f"🔥 Detonated {initial - len(self.data)} statistical anomalies out of dataset.")

        return self.data

    def extract_normalized_numeric_data(self, scaling: str = 'minmax', columns: Optional[List[str]] = None) -> pd.DataFrame:
        if columns is None:
            columns = self.numeric_cols
        if not columns:
            return pd.DataFrame()

        scalers = {'minmax': MinMaxScaler(), 'standard': StandardScaler(), 'robust': RobustScaler()}
        scaler = scalers.get(scaling)
        if not scaler:
            raise ValueError(f"Invalid scaling technique choice: {list(scalers.keys())}")

        scaled_data = scaler.fit_transform(self.data[columns])
        scaled_df = pd.DataFrame(scaled_data, columns=[f"{col}_scaled" for col in columns], index=self.data.index)
        print(f"⚡ Normalized Scaler Engineered via [{scaling.upper()}] over {len(columns)} dimensions.")
        return scaled_df

    def extract_normalized_categorical_data(self, encoding: str = 'onehot', columns: Optional[List[str]] = None, target_col: Optional[str] = None) -> pd.DataFrame:
        if columns is None:
            columns = self.categorical_cols
        if not columns:
            return pd.DataFrame()

        if encoding == 'onehot':
            encoded = pd.get_dummies(self.data[columns], prefix=columns, drop_first=True)
            # Standardize output column formatting to look majestic
            encoded = encoded.astype(float)
            print(f"🎭 High-Dimensional Encoded Node Set: Created {encoded.shape[1]} Hot Vectors.")
            return encoded

        elif encoding == 'ordinal':
            encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            encoded_array = encoder.fit_transform(self.data[columns].astype(str))
            print(f"🎭 Sequential Mapping: Ordinal Vectors compiled across {len(columns)} attributes.")
            return pd.DataFrame(encoded_array, columns=columns, index=self.data.index)

        elif encoding == 'target' and target_col is not None:
            # High level Master-Class feature engineering mapping technique
            target_encoded_df = pd.DataFrame(index=self.data.index)
            for col in columns:
                means = self.data.groupby(col)[target_col].transform('mean')
                target_encoded_df[f"{col}_target_encoded"] = means
            print(f"🎯 Elite Mode: Target Encoding formulated mapping against target: [{target_col}]")
            return target_encoded_df
        else:
            raise ValueError("Encoding parameters misaligned. Try 'onehot', 'ordinal', or 'target' (with target_col configuration).")

    def merge_normalized_data(self, numeric_df: Optional[pd.DataFrame] = None, categorical_df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
        result = self.data.copy()
        if numeric_df is not None and not numeric_df.empty:
            result = pd.concat([result, numeric_df], axis=1)
        if categorical_df is not None and not categorical_df.empty:
            result = pd.concat([result, categorical_df], axis=1)
        print(f"🔮 Unified Sovereign Tensor Compiled Successfully. Master Shape Matrix: {result.shape}")
        return result

    def plot_univariate_subplots(self, columns: Optional[List[str]] = None) -> None:
        if columns is None:
            columns = self.numeric_cols[:3]

        for col in columns:
            fig = make_subplots(rows=1, cols=3, subplot_titles=[f"🎻 {col} Distribution Density", f"🌌 {col} Spatial Scatter", f"📊 {col} Standard Bin Count"])

            fig.add_trace(go.Violin(y=self.data[col], name=col, box_visible=True, line_color=LuxuryTheme.PRIMARY, fillcolor=LuxuryTheme.SECONDARY, opacity=0.8), row=1, col=1)
            fig.add_trace(go.Scatter(x=self.data.index, y=self.data[col], mode='markers', marker=dict(size=5, color=self.data[col], colorscale=LuxuryTheme.SEQUENTIAL, showscale=False)), row=1, col=2)
            fig.add_trace(go.Histogram(x=self.data[col], nbinsx=30, marker_color=LuxuryTheme.ACCENT, marker_line=dict(color=LuxuryTheme.BACKGROUND, width=1)), row=1, col=3)

            fig = LuxuryTheme.apply_layout(fig, f"Univariate Analysis: {col}")
            fig.update_layout(height=450, showlegend=False)
            fig.show()

    def plot_relationship(self, col1: str, col2: str) -> None:
        type1 = 'numeric' if col1 in self.numeric_cols else 'categorical'
        type2 = 'numeric' if col2 in self.numeric_cols else 'categorical'

        if type1 == 'numeric' and type2 == 'numeric':
            fig = px.scatter(self.data, x=col1, y=col2, trendline='ols', trendline_color_override=LuxuryTheme.SECONDARY,
                             color=col2, color_continuous_scale=LuxuryTheme.SEQUENTIAL)
            fig = LuxuryTheme.apply_layout(fig, f"Bivariate Continuum: {col1} vs {col2}")
            fig.show()

        elif (type1 == 'numeric' and type2 == 'categorical') or (type1 == 'categorical' and type2 == 'numeric'):
            num_col = col1 if type1 == 'numeric' else col2
            cat_col = col2 if type1 == 'numeric' else col1
            fig = px.box(self.data, x=cat_col, y=num_col, color=cat_col, points='all')
            fig = LuxuryTheme.apply_layout(fig, f"Category Breakdown: {num_col} mapped by {cat_col}")
            fig.show()

        else:
            contingency = pd.crosstab(self.data[col1], self.data[col2])
            fig = px.bar(contingency, barmode='group')
            fig = LuxuryTheme.apply_layout(fig, f"Categorical Fusion Intersect: {col1} x {col2}")
            fig.show()

    def plot_categorical_frequency(self, column: str, top_n: int = 10) -> None:
        freq = self.data[column].value_counts().head(top_n)
        pct = (freq / len(self.data) * 100).round(1)

        fig = go.Figure(data=[
            go.Bar(x=freq.index.astype(str), y=freq.values,
                   marker=dict(color=freq.values, colorscale=LuxuryTheme.SEQUENTIAL),
                   text=[f"🔥 Count: {v}<br>📊 ({p}%)" for v, p in zip(freq.values, pct)],
                   textposition='outside')
        ])
        fig = LuxuryTheme.apply_layout(fig, f"Frequency Distribution Vector Matrix: {column}")
        fig.update_layout(yaxis_title="Raw Accumulative Volumetric Scale")
        fig.show()

    def plot_all_associations_heatmap(self) -> None:
        """Computes true statistical associations: Pearson (Num-Num), Cramér's V (Cat-Cat), Correlation Ratio (Num-Cat)"""
        all_cols = self.numeric_cols + self.categorical_cols
        n = len(all_cols)
        association_matrix = np.zeros((n, n))

        # Advanced Mathematics Engine for Statistical Tracking
        for i, col1 in enumerate(all_cols):
            for j, col2 in enumerate(all_cols):
                if i == j:
                    association_matrix[i, j] = 1.0
                elif i > j:
                    association_matrix[i, j] = association_matrix[j, i]
                else:
                    # 1. Pearson Numerical Correlation
                    if col1 in self.numeric_cols and col2 in self.numeric_cols:
                        corr_val = self.data[col1].corr(self.data[col2])
                        association_matrix[i, j] = corr_val if not np.isnan(corr_val) else 0.0

                    # 2. Cramér's V Advanced Categorical Metric Calculation
                    elif col1 in self.categorical_cols and col2 in self.categorical_cols:
                        contingency = pd.crosstab(self.data[col1], self.data[col2])
                        if contingency.size > 0 and contingency.sum().sum() > 0:
                            chi2 = stats.chi2_contingency(contingency)[0]
                            n_total = contingency.sum().sum()
                            min_dim = min(contingency.shape) - 1
                            association_matrix[i, j] = np.sqrt(chi2 / (n_total * min_dim)) if min_dim > 0 else 0.0
                        else:
                            association_matrix[i, j] = 0.0

                    # 3. Numerical & Categorical Fusion Association ($\eta$ calculation)
                    else:
                        num_col = col1 if col1 in self.numeric_cols else col2
                        cat_col = col2 if col1 in self.numeric_cols else col1

                        # Calculate categorical target means vs global variances
                        global_mean = self.data[num_col].mean()
                        category_means = self.data.groupby(cat_col)[num_col].mean()
                        category_counts = self.data.groupby(cat_col)[num_col].count()

                        ss_between = (category_counts * (category_means - global_mean) ** 2).sum()
                        ss_total = ((self.data[num_col] - global_mean) ** 2).sum()

                        eta = np.sqrt(ss_between / ss_total) if ss_total > 0 else 0.0
                        association_matrix[i, j] = eta

        fig = px.imshow(association_matrix, x=all_cols, y=all_cols,
                        color_continuous_scale=LuxuryTheme.DIVERGING, text_auto='.2f', zmin=-1.0, zmax=1.0)
        fig = LuxuryTheme.apply_layout(fig, "Universal Quantum Association Matrix Heatmap")
        fig.update_layout(height=max(600, 40 * n), width=max(600, 40 * n))
        fig.show()


# ==========================================
# TESTBENCH DEMONSTRATION ENGINE
# ==========================================
if __name__ == "__main__":
    print("="*70)
    print("⚜️ MASTER ENGINE CORE INITIALIZED - SYSTEM RUNNING SEAMLESSLY ⚜️")
    print("="*70)

    inspector = DataInspector()

    print("\n🌐 Injecting Production Stream: Titanic Core Payload...")
    titanic_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    inspector.data = pd.read_csv(titanic_url)
    inspector.original_data = inspector.data.copy()
    inspector._identify_column_types()

    # Pipeline Processing Actions
    inspector.data_summary()
    inspector.handle_missing_values(strategy='knn') # Swapped to advanced KNN imputer
    inspector.remove_duplicates()
    inspector.handle_outliers(columns=['Age', 'Fare'], method='flag')

    # Advanced Multi-Encoding Operations
    numeric_scaled = inspector.extract_normalized_numeric_data(scaling='robust', columns=['Age', 'Fare'])
    categorical_encoded = inspector.extract_normalized_categorical_data(encoding='target', columns=['Sex', 'Embarked'], target_col='Survived')

    merged = inspector.merge_normalized_data(numeric_scaled, categorical_encoded)

    # Launching Beautiful Visual Analytical Frameworks
    print("\n🎭 Plotting Majestic Dashboard Ecosystem...")
    inspector.plot_univariate_subplots(['Age', 'Fare'])
    inspector.plot_relationship('Age', 'Fare')
    inspector.plot_relationship('Sex', 'Survived')
    inspector.plot_categorical_frequency('Pclass')
    inspector.plot_all_associations_heatmap()

    plotter = PlottingMethods()
    print("\n✨ Process finished cleanly. Final Sovereign Architecture Shape Compiled Vector:", inspector.data.shape)

⚜️ MASTER ENGINE CORE INITIALIZED - SYSTEM RUNNING SEAMLESSLY ⚜️

🌐 Injecting Production Stream: Titanic Core Payload...

👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 
                       MAJESTIC DATA SUMMARY REPORT
👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 👑 
📈 Total Rows: 891 | Total Columns: 12
🔢 Numeric Elements: 7 | 🔤 Categorical Features: 5

📥 METADATA OBSERVATION MATRIX (FIRST 5 ROWS):
|    |   PassengerId |   Survived |   Pclass | Name                                                | Sex    |   Age |   SibSp |   Parch | Ticket           |    Fare | Cabin   | Embarked   |
|---:|--------------:|-----------:|---------:|:----------------------------------------------------|:-------|------:|--------:|--------:|:-----------------|--------:|:--------|:-----------|
|  0 |             1 |          0 |        3 | Braund, Mr. Owen Harris                             | male   |    22 |       1 |       0 | A/5 21171        |  7.25   | nan     | S          |
|  1 |             2 |          1 |    


✨ Process finished cleanly. Final Sovereign Architecture Shape Compiled Vector: (891, 13)
